# Hooks 钩子机制教程 (v0.8)

本教程帮助你理解 Agent 系统中**钩子（Hooks）**的概念和使用方式。

## 学习目标

1. 理解钩子机制：在特定事件点注册和触发回调
2. 掌握 HookEvent 和 HookContext：事件类型与数据传递
3. 学会优先级排序和短路机制
4. 了解钩子在 Agent 生命周期中的实际应用

In [ ]:
# Step 1: 安装依赖
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "openai", "python-dotenv", "zhipuai", "-q"
])
print("依赖安装完成！")

In [ ]:
# Step 2: 设置项目路径
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    project_root = cwd.parent
else:
    project_root = cwd
    while not (project_root / "pyproject.toml").exists():
        project_root = project_root.parent

src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"项目根目录: {project_root}")
print(f"源码路径: {src_path}")

## 1. 什么是钩子？

钩子（Hook）是一种**在不修改核心逻辑的情况下扩展系统行为**的机制。

| 扩展方式 | 优点 | 缺点 |
|---------|------|------|
| 继承 | 简单直接 | 耦合度高，修改影响大 |
| 装饰器 | 包装函数 | 难以动态组合和移除 |
| **钩子** | **运行时可动态调整** | 需要设计事件体系 |

在 Agent 系统中，钩子特别适合处理**横切关注点**：
- 日志记录
- 性能计时
- 权限检查
- 错误处理

In [ ]:
# 导入 Hook 相关类型
from rogue_rabbit.contracts.hook import HookEvent, HookContext, HookCallback
from rogue_rabbit.core.hook_manager import HookManager

# 创建钩子管理器
manager = HookManager()

print("导入成功！")
print(f"HookEvent 有 {len(HookEvent)} 个事件类型:")
for event in HookEvent:
    print(f"  - {event.value}")

## 2. HookEvent 和 HookContext

### HookEvent - 生命周期事件

```
ON_START → BEFORE_LLM_CALL → AFTER_LLM_CALL
        → BEFORE_TOOL_CALL → AFTER_TOOL_CALL
        → ON_ERROR (异常时)
        → ON_FINISH
```

### HookContext - 钩子上下文

| 属性 | 说明 |
|------|------|
| `event` | 触发的事件类型 |
| `data` | 事件相关数据（可读写） |
| `stopped` | 是否短路（停止后续钩子） |

In [ ]:
# 理解 HookContext
ctx = HookContext(
    event=HookEvent.BEFORE_LLM_CALL,
    data={"prompt": "你好", "model": "glm-4"}
)

print(f"事件: {ctx.event.value}")
print(f"数据: {ctx.data}")
print(f"已停止: {ctx.stopped}")

# 钩子可以修改 data
ctx.data["temperature"] = 0.7
print(f"\n修改后数据: {ctx.data}")

# 钩子可以设置短路
ctx.stopped = True
print(f"设置短路: {ctx.stopped}")

## 3. 基础用法 — 注册、触发、卸载

In [ ]:
# 注册和触发钩子
manager = HookManager()

def logging_hook(ctx: HookContext) -> HookContext:
    """日志钩子：记录事件数据"""
    print(f"  [日志] {ctx.event.value}: {ctx.data}")
    return ctx

# 注册
hook_id = manager.register(HookEvent.BEFORE_LLM_CALL, logging_hook)
print(f"注册钩子: {hook_id}")

# 触发
print("\n触发 BEFORE_LLM_CALL:")
ctx = HookContext(event=HookEvent.BEFORE_LLM_CALL, data={"prompt": "hello"})
result = manager.trigger(HookEvent.BEFORE_LLM_CALL, ctx)

# 卸载
success = manager.unregister(hook_id)
print(f"\n卸载钩子 {hook_id}: {'成功' if success else '失败'}")

# 再次触发（无钩子）
print("\n卸载后触发（无输出）:")
manager.trigger(HookEvent.BEFORE_LLM_CALL, HookContext(event=HookEvent.BEFORE_LLM_CALL))

## 4. 优先级与短路

### 优先级
- `priority` 越大越先执行
- 默认 priority = 0

### 短路
- 设置 `context.stopped = True` 跳过后续钩子
- 适合做权限守卫、输入验证等

In [ ]:
# 优先级排序
manager2 = HookManager()
order = []

def make_hook(name: str):
    def hook(ctx: HookContext) -> HookContext:
        order.append(name)
        return ctx
    return hook

manager2.register(HookEvent.ON_START, make_hook("日志(0)"), priority=0)
manager2.register(HookEvent.ON_START, make_hook("权限(10)"), priority=10)
manager2.register(HookEvent.ON_START, make_hook("限流(5)"), priority=5)

manager2.trigger(HookEvent.ON_START, HookContext(event=HookEvent.ON_START))
print(f"执行顺序: {' → '.join(order)}")
print("\n要点: priority 越大越先执行")

In [ ]:
# 短路机制
manager3 = HookManager()
order2 = []

def auth_guard(ctx: HookContext) -> HookContext:
    order2.append("auth_guard")
    if not ctx.data.get("token"):
        print("  [守卫] 无 token，拦截!")
        ctx.stopped = True
    return ctx

def executor(ctx: HookContext) -> HookContext:
    order2.append("executor")
    return ctx

manager3.register(HookEvent.BEFORE_TOOL_CALL, auth_guard, priority=10)
manager3.register(HookEvent.BEFORE_TOOL_CALL, executor, priority=0)

# 无 token → 短路
print("场景 1: 无 token")
order2.clear()
ctx = manager3.trigger(HookEvent.BEFORE_TOOL_CALL, HookContext(event=HookEvent.BEFORE_TOOL_CALL))
print(f"  执行: {' → '.join(order2)}, stopped={ctx.stopped}")

# 有 token → 正常
print("\n场景 2: 有 token")
order2.clear()
ctx2 = HookContext(event=HookEvent.BEFORE_TOOL_CALL, data={"token": "valid"})
ctx2 = manager3.trigger(HookEvent.BEFORE_TOOL_CALL, ctx2)
print(f"  执行: {' → '.join(order2)}, stopped={ctx2.stopped}")

## 5. Agent 生命周期集成

模拟 Agent 执行循环，在各生命周期节点触发钩子。

In [ ]:
class SimpleAgent:
    """简化版 Agent，演示钩子在生命周期中的触发"""
    
    def __init__(self, hook_manager: HookManager) -> None:
        self.hooks = hook_manager
    
    def run(self, question: str) -> str:
        # ON_START
        ctx = HookContext(event=HookEvent.ON_START, data={"question": question})
        self.hooks.trigger(HookEvent.ON_START, ctx)
        
        # BEFORE_LLM_CALL
        ctx = HookContext(event=HookEvent.BEFORE_LLM_CALL, data={"prompt": question})
        self.hooks.trigger(HookEvent.BEFORE_LLM_CALL, ctx)
        
        # 模拟 LLM 响应
        llm_response = "思考: 需要计算"
        
        # AFTER_LLM_CALL
        ctx = HookContext(event=HookEvent.AFTER_LLM_CALL, data={"response": llm_response})
        self.hooks.trigger(HookEvent.AFTER_LLM_CALL, ctx)
        
        # BEFORE_TOOL_CALL
        ctx = HookContext(event=HookEvent.BEFORE_TOOL_CALL, data={"tool": "calculator", "args": "2+3"})
        self.hooks.trigger(HookEvent.BEFORE_TOOL_CALL, ctx)
        
        # 模拟工具执行
        tool_result = "5"
        
        # AFTER_TOOL_CALL
        ctx = HookContext(event=HookEvent.AFTER_TOOL_CALL, data={"tool": "calculator", "result": tool_result})
        self.hooks.trigger(HookEvent.AFTER_TOOL_CALL, ctx)
        
        # ON_FINISH
        answer = f"答案是 {tool_result}"
        ctx = HookContext(event=HookEvent.ON_FINISH, data={"answer": answer})
        self.hooks.trigger(HookEvent.ON_FINISH, ctx)
        
        return answer


# 创建带钩子的 Agent
manager4 = HookManager()
flow = []

def trace_hook(ctx: HookContext) -> HookContext:
    flow.append(ctx.event.value)
    return ctx

for event in HookEvent:
    if event != HookEvent.ON_ERROR:
        manager4.register(event, trace_hook)

agent = SimpleAgent(manager4)
result = agent.run("计算 2+3")

print(f"生命周期流: {' → '.join(flow)}")
print(f"结果: {result}")

## 6. 实用组合 — 日志 + 计时 + 权限

实际场景中，多个钩子协同工作：
- 权限守卫（高优先级）：拦截未授权请求
- 计时钩子（中优先级）：记录耗时
- 日志钩子（低优先级）：记录操作

In [ ]:
import time

manager5 = HookManager()

# 权限守卫（最高优先级）
def permission_guard(ctx: HookContext) -> HookContext:
    if ctx.data.get("user_role") == "guest":
        print("  [权限] 拒绝 guest 用户")
        ctx.stopped = True
        return ctx
    print(f"  [权限] 允许 {ctx.data.get('user_role')} 用户")
    return ctx

# 计时钩子
timer = {"start": 0.0}

def start_timer(ctx: HookContext) -> HookContext:
    timer["start"] = time.time()
    return ctx

def end_timer(ctx: HookContext) -> HookContext:
    elapsed = (time.time() - timer["start"]) * 1000
    print(f"  [计时] 耗时 {elapsed:.1f}ms")
    return ctx

# 日志钩子（最低优先级）
def log_hook(ctx: HookContext) -> HookContext:
    print(f"  [日志] {ctx.event.value}: {list(ctx.data.keys())}")
    return ctx

# 注册前置钩子
manager5.register(HookEvent.BEFORE_TOOL_CALL, permission_guard, priority=20)
manager5.register(HookEvent.BEFORE_TOOL_CALL, start_timer, priority=10)
manager5.register(HookEvent.BEFORE_TOOL_CALL, log_hook, priority=0)

# 注册后置钩子
manager5.register(HookEvent.AFTER_TOOL_CALL, log_hook, priority=10)
manager5.register(HookEvent.AFTER_TOOL_CALL, end_timer, priority=0)

# 场景 1: admin
print("=== 场景: admin 用户 ===")
ctx = HookContext(event=HookEvent.BEFORE_TOOL_CALL, data={"tool": "search", "user_role": "admin"})
result = manager5.trigger(HookEvent.BEFORE_TOOL_CALL, ctx)
if not result.stopped:
    result.data["result"] = "搜索结果"
    manager5.trigger(HookEvent.AFTER_TOOL_CALL, result)

# 场景 2: guest（被拦截）
print("\n=== 场景: guest 用户 ===")
ctx2 = HookContext(event=HookEvent.BEFORE_TOOL_CALL, data={"tool": "search", "user_role": "guest"})
result2 = manager5.trigger(HookEvent.BEFORE_TOOL_CALL, ctx2)
if not result2.stopped:
    result2.data["result"] = "搜索结果"
    manager5.trigger(HookEvent.AFTER_TOOL_CALL, result2)
else:
    print("  [结果] 请求被拦截")

## 总结

### 核心概念

| 概念 | 说明 |
|------|------|
| `HookEvent` | Agent 生命周期事件类型（7 个） |
| `HookContext` | 钩子上下文（事件 + 数据 + 短路标志） |
| `HookCallback` | 钩子回调函数，返回 None 或修改后的 context |
| `HookManager` | 管理钩子的注册、触发和卸载 |

### 设计原则

| 原则 | 说明 |
|------|------|
| 洋葱模型 | 请求穿过多层钩子，每层可拦截或修改 |
| 正交扩展 | 不修改核心逻辑即可添加行为 |
| 顺序控制 | priority 越大越先执行 |
| 短路机制 | stopped=True 跳过后续钩子 |

### 下一步

- 运行 `experiments/21_hook_basic.py` 基础钩子
- 运行 `experiments/22_hook_lifecycle.py` 生命周期钩子
- 运行 `experiments/23_hook_chain.py` 钩子链与优先级